In [22]:
import pandas as pd
import numpy as np
import json

# State Level data

In [2]:
# get 50 states + DC for all datasets
locations = pd.read_csv('../data/location.csv').iloc[:52]
# convert population to be in units of 10,000
locations["population"] = locations["population"]/1e4
# make dataframe holding population of each state in columns
pop_data = locations.loc[:,["location","population"]].T
pop_data.columns = pop_data.iloc[0].str.lstrip('0')
pop_data = pop_data[1:].reset_index(drop=True)

### Normalize and Format Covid-19/Flu datasets

In [5]:
def format_normalize_dataset(df, population_df, ):
    df = df[locations.location.values]
    # Remove leading zeros, for consistency with other merges
    df.columns = df.columns.str.lstrip('0')
    df = df.drop('US', axis=1)
    # rearrange columns to be the same for both datasets, divide column-wise
    population_df = population_df[df.columns]
    return df.divide(population_df.iloc[0], axis=1)
    

In [6]:
# Covid-19 Confirmed Cases per 10k ppl
cases = pd.read_csv('../data/covid19_incident_cases.csv',parse_dates=['date']).set_index('date')
normalized_cases = format_normalize_dataset(cases, pop_data)
normalized_cases.to_csv('../data/covid19_incident_cases_normalized.csv')

In [7]:
# Covid-19 Hospitalizations per 10k ppl
hosp = pd.read_csv('../data/covid19_incident_hosp.csv',parse_dates=['date']).set_index('date')
normalized_hosp = format_normalize_dataset(hosp, pop_data)
normalized_hosp.to_csv('../data/covid19_incident_hosp_normalized.csv')

In [8]:
# Influenza Hospitalizations per 10k ppl
flu_hosp = pd.read_csv('../data/flu_incident_hosp.csv',parse_dates=['date']).set_index('date')
normalized_flu_hosp = format_normalize_dataset(flu_hosp, pop_data)
normalized_flu_hosp.to_csv('../data/flu_incident_hosp_normalized.csv')

### Get State Population Centers

In [258]:
# Data from 2020 US Census
src = "https://www2.census.gov/geo/docs/reference/cenpop2020/CenPop2020_Mean_ST.txt"
centers = pd.read_csv(src)
centers["STATEFP"] = centers["STATEFP"].astype(str)
fips_df = pd.DataFrame(centers["STATEFP"])
centers_df = (fips_df.merge(fips_df, how="cross") # cartesian product of state FIPS codes
              # get latitude, longitude of origin state
                     .merge(centers.loc[:,["STATEFP","LATITUDE", "LONGITUDE"]],
                            left_on="STATEFP_x", right_on="STATEFP", how="left")
                     .drop(columns=["STATEFP"])
              # get latitude, longitude of destination state
                     .merge(centers.loc[:,["STATEFP","LATITUDE", "LONGITUDE"]],
                            left_on="STATEFP_y", right_on="STATEFP", how="left")
                     .drop(columns=["STATEFP"])
              # make column names more descriptive
                     .rename(columns={
                         "STATEFP_x": "origin",
                         "STATEFP_y": "destination",
                         "LATITUDE_x": "origin_lat",
                         "LONGITUDE_x": "origin_long",
                         "LATITUDE_y": "destination_lat",
                         "LONGITUDE_y": "destination_long"
                         }))

### Calculate distance between state population centers

In [32]:
def haversine_distance(lat1, long1, lat2, long2,decimals=2):
    """
    Distance (in km) between two pairs of latitude, longitude coordinates according to haversine 
    formula.
    """
    # convert all coords to radians
    lat1, long1, lat2, long2 = map(np.radians, [lat1, long1, lat2, long2])
    # earth's radius (km)
    r =  6378.137
    # inside sqrt part of haversine formula
    inside_sqrt = np.sin((lat2-lat1)/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin((long2-long1)/2)**2
    return  round(2*r*np.arcsin(np.sqrt(inside_sqrt)),decimals) 

In [274]:
centers_df['distance'] = haversine_distance(centers_df['origin_lat'],
                                           centers_df['origin_long'],
                                           centers_df['destination_lat'],
                                           centers_df['destination_long']
                                           )
# remove self loops
centers_df = centers_df.loc[centers_df.origin != centers_df.destination,:]
centers_formatted = centers_df.loc[:,["origin","destination","distance"]]
centers_formatted.to_csv('../data/state_populationcenters_distance.csv', index=False)

### Process (state) Patchflow Datasets
Reformat unnormalized flow datasets for total (bidirectional) flow between states and organize into 3 column format needed for STEpPE to make spatial embeddings

In [266]:
# Patchflow uses GADM codes not FIPS codes, need to convert
# Dict from https://gadm.org/maps/USA_1.html
GADM_codes = {'GADM': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 
                       17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 
                       31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 
                       45, 46, 47, 48, 49, 50, 51], 
              'location_name': ['Maine', 'Washington', 'North Dakota', 'Montana', 'Vermont', 
                       'New Hampshire', 'Minnesota', 'Massachusetts', 'Oregon', 
                       'Michigan', 'New York', 'Rhode Island', 'Idaho', 'Wisconsin', 
                       'Connecticut', 'South Dakota', 'Wyoming', 'New Jersey', 
                       'Pennsylvania', 'Iowa', 'Ohio', 'Delaware', 'Maryland', 
                       'Nebraska', 'District of Columbia', 'Indiana', 'Illinois', 
                       'Nevada', 'West Virginia', 'Utah', 'Virginia', 'Colorado', 
                       'California', 'Missouri', 'Kentucky', 'Kansas', 'North Carolina', 
                       'Tennessee', 'South Carolina', 'Oklahoma', 'Arkansas', 'Arizona', 
                       'New Mexico', 'Georgia', 'Alabama', 'Mississippi', 'Texas', 
                       'Louisiana', 'Florida', 'Alaska', 'Hawaii']
}
# make datafrmae containing all states and their respective GADM and FIPS codes
gadm_df = pd.DataFrame(GADM_codes)
converter_df = (centers.loc[:,["STATEFP","STNAME"]].
                rename(columns={
                    "STATEFP":"FIPS",
                    "STNAME":"location_name"}).
                # merge dataset of statenames and fips codes with GADM dataset
                merge(gadm_df, on="location_name", how="left")
               )
# convert GADM codes to str and fill na values with 0 (puerto rico in FIPS data not in GADM data)
converter_df["GADM"] = converter_df["GADM"].fillna(0)
converter_df["GADM"] = converter_df["GADM"].astype(int).astype(str)
# GADM to FIPS lookup table
GADM_to_FIPS = converter_df.loc[:,["GADM","FIPS"]].set_index("GADM").to_dict()["FIPS"]

In [35]:
# Patch flow data has flow with direction, use total birdirectional flow for undirected metric
def get_total_flow(flow_df, conversion_dict, clean_codes=True):
    """
    Gets the total flow between two locations, according to radiation model, by 
    summing the flow to and from a location. Not normalized by population
    """
    if clean_codes:
        # Get the GADM code without extra characters
        flow_df['origin'] = (flow_df['origin'].str.replace('USA.', '').
                             str.replace('_1', ''))
        flow_df['destination'] = (flow_df['destination'].str.replace('USA.', '').
                             str.replace('_1', ''))
    # remove self loops
    flow_df = flow_df.loc[flow_df.origin != flow_df.destination,:]
    # If original data has flow from A -> B, reverse flow is B -> A
    flow_reversed = flow_df.rename(columns={'origin': 'destination', 
                                            'destination': 'origin', 
                                            'flow': 'reverse_flow'})
    # outer merge: 
    #if there is B -> C but not C -> B, still want to include a row C-> B in the final undirected data 
    flow_merged = pd.merge(flow_df, flow_reversed, on=['origin', 'destination'], 
                           how='outer')
    # Fill NAs with 0 for no flow
    flow_merged['flow'] = flow_merged['flow'].fillna(0)
    flow_merged['reverse_flow'] = flow_merged['reverse_flow'].fillna(0)
    # Get bi-directional flow
    flow_merged['total_flow'] = flow_merged['flow'] + flow_merged['reverse_flow']
    
    # Convert to FIPS, drop directional flow columns
    final_df = flow_merged.drop(['flow','reverse_flow'], axis=1)
    final_df['origin'] = final_df['origin'].map(conversion_dict) 
    final_df['destination'] = final_df['destination'].map(conversion_dict) 
    return final_df

In [270]:
# patchflow admin level 1 data cloned from:
# https://github.com/NSSAC/patchflow-data/tree/main/data/v1.0/USA
rad_constants = [0.01, 0.02, 0.05, 0.1, 0.2, 0.5]
for r in rad_constants:
    src = "../data/patchflow_raw/USA_admin1_radiation_constant_{:.02f}.csv".format(r)
    patch_df = pd.read_csv(src).drop("time", axis=1)
    flow_data = get_total_flow(patch_df, GADM_to_FIPS)
    path = "../data/flows_processed/USA_admin1_radiation_constant_{:.02f}_processed.csv".format(r)
    flow_data.to_csv(path, index=False)

# County Level data

In [19]:
# 2023-24 weekly covid-19 hospitalizations by county

# data from https://data.cdc.gov/Public-Health-Surveillance/Weekly-United-States-COVID-19-Hospitalization-Metr/akn2-qxic/data_preview
hosp_county_2324 = (pd.read_csv(
    "../data/COVID-19_Hosp_by_County_2023-24.csv",
    parse_dates=["report_date", "week_end_date"])
    .set_index("week_end_date")
    .loc[:,["state", "county", "fips_code", "county_population", 
            "total_adm_all_covid_confirmed_past_7days_per_100k",
            ]]
    )

# add back leading zeros of FIPS code
hosp_county_2324["fips_code"] = hosp_county_2324["fips_code"].astype(str).str.zfill(5)

# make some column names shorter
hosp_county_2324 = hosp_county_2324.rename(columns={"total_adm_all_covid_confirmed_past_7days_per_100k":"hosp_per_100k", "county_population":"population"})

# reshape to wide format with rows as date (weeks in 2023-2024), coumns as county fips codes, values 
reshaped_2324 = hosp_county_2324.reset_index().pivot(
    index='week_end_date', 
    columns='fips_code', 
    values='hosp_per_100k'
)

In [20]:
# 2022-2023 weekly covid-19 hospitalizations by county
# data from https://healthdata.gov/dataset/United-States-COVID-19-Community-Levels-by-County/nn5b-j5u9/about_data

hosp_county_2223 = (pd.read_csv(
    "../data/COVID-19_Hosp_by_County_2022-23.csv",
    parse_dates=["date_updated"])
    .set_index("date_updated")
    .loc[:,["state", "county", "county_fips", "county_population", 
            "covid_hospital_admissions_per_100k",
            ]]
    ).rename(columns={"county_fips":"fips_code"})

# add back leading zeros of FIPS code
hosp_county_2223["fips_code"] = hosp_county_2223["fips_code"].astype(str).str.zfill(5)

# make column names consistent with 2023-24 dataset
hosp_county_2223 = hosp_county_2223.rename(columns={"covid_hospital_admissions_per_100k":"hosp_per_100k", "county_population":"population"})

# reshape to wide format with rows as date (weeks in 2023-2024), coumns as county fips codes, values 
reshaped_2223 = hosp_county_2223.reset_index().pivot(
    index='date_updated', 
    columns='fips_code', 
    values='hosp_per_100k'
)
# round to nearest saturday for consistency with 2023-24 data
reshaped_2223.index = reshaped_2223.index + pd.offsets.Week(weekday=5)
# filter to only dates before the 2023-24 dataset
reshaped_2223 = reshaped_2223.loc[reshaped_2223.index < '2023-05-06']


In [21]:
# combine 2022-23 and 2023-24 datasets
hosp_county_2224 = pd.concat([reshaped_2223, reshaped_2324], axis=0)

# write to parquet
hosp_county_2224.to_parquet( "../data/COVID-19_Hosp_by_County_2022-24.parquet", index=False)

### Make a conversion table for GADM to FIPS codes

Unlike the state level data, there isn't a readily available conversion table. We need to make our own.

In [26]:
# contains all county gadm codes and the county/state names
with open("../data/gadm_USA_2.json", "r") as f:
    data = json.load(f)
    
# Extract relevant info: GADM code, state name, county name
records = []
for feature in data["features"]:
    props = feature["properties"]
    records.append({
        "state_name": props["NAME_1"],
        "county_name": props["NAME_2"],
        "gadm_code": props["GID_2"]
    })

# Convert to DataFrame
gadm_df = pd.DataFrame(records)
# remove spaces and convert to all lower case for consistency
gadm_df["state_name"] = gadm_df["state_name"].str.replace(' ', '', regex=False).str.lower()

In [25]:
# contains all fips codes, county/state names, and lat/long coordinates

# lat/long of population center by county
# from https://www.census.gov/geographies/reference-files/time-series/geo/centers-population.html
county_centers = pd.read_csv("../data/county_pop_centers.txt")
# add back leading zeros of FIPS codes
county_centers["STATEFP"] = county_centers["STATEFP"].astype(str).str.zfill(2)
county_centers["COUNTYFP"] = county_centers["COUNTYFP"].astype(str).str.zfill(3)

# remove spaces and convert to all lowercase for consistency
county_centers["STNAME"] = county_centers["STNAME"].str.replace(' ', '').str.lower()
county_centers["COUNAME"] = county_centers["COUNAME"].str.replace(' ', '').str.lower()

In [ ]:
# Manually adjust some county names in the GADM data to merge on the FIPS data:

# convert "Saint" to abbreviation "st."
gadm_df["county_name"] = gadm_df["county_name"].str.replace("Saint", "st.", regex=False).str.replace(' ', '', regex=False).str.lower()

# adjust these names for consistent spelling with FIPS data
gadm_df.loc[(gadm_df["state_name"] == "missouri") & (gadm_df["county_name"]=="st.egenevieve"), "county_name"] = "ste.genevieve"
gadm_df.loc[(gadm_df["state_name"] == "newmexico") & (gadm_df["county_name"]=="donaana"), "county_name"] = "doñaana"

# wrangell-petersburg is now 2 different census areas, just use petersburg as placeholder
gadm_df.loc[(gadm_df["state_name"] == "alaska") & (gadm_df["county_name"]=="wrangell-petersburg"), "county_name"] = "petersburg"

# gadm condenses skagway, yakutat, and angoon into one location, rename this condensed version to skagway
gadm_df.loc[(gadm_df["state_name"] == "alaska") & (gadm_df["county_name"]=="skagway-yakutat-angoon"), "county_name"] = "skagway"

# gadm uses the old name for the prince of wales-hyder census area (used to be outer ketchikan)
gadm_df.loc[(gadm_df["state_name"] == "alaska") & (gadm_df["county_name"]=="princeofwales-outerketchi"), "county_name"] = "princeofwales-hyder"

# wade hampton is now known as kusilvak
gadm_df.loc[(gadm_df["state_name"] == "alaska") & (gadm_df["county_name"]=="wadehampton"), "county_name"] = "kusilvak"

# shannon is now oglala lakota county
gadm_df.loc[(gadm_df["state_name"] == "southdakota") & (gadm_df["county_name"]=="shannon"), "county_name"] = "oglalalakota"

# valdez cordova split into chugach and copper river, will just use chugach since most of the population lives there
gadm_df.loc[(gadm_df["state_name"] == "alaska") & (gadm_df["county_name"]=="valdez-cordova"), "county_name"] = "chugach"

In [27]:
# merge fips and gadm codes based on county and state names
fips_df = county_centers.rename(columns={"COUNAME":"county_name", "STNAME":"state_name"})
gadm2fips = pd.merge(gadm_df, fips_df, how="left", 
                     on=["state_name", "county_name"]).dropna() # drops 15 cases, 11 corresponding to lakes and 4 small incorporated cities in virginia

gadm2fips["fips"] = gadm2fips["STATEFP"] + gadm2fips["COUNTYFP"]

This dictionary will be used to convert the GADM codes in the Patchflow datasets to FIPS codes

In [30]:
# make a dictionary mapping gadm codes to fips
county_conversion_dict = gadm2fips.loc[:,["gadm_code","fips"]].set_index("gadm_code").to_dict()["fips"]

### Calculate Pairwise distances between counties

In [33]:
lat_long_df = gadm2fips.loc[:,["fips", "LATITUDE", "LONGITUDE"]]

# reformat to pass data into the haversine_distance function from before
county_distances = (lat_long_df.merge(lat_long_df, how="cross").
                    rename(columns={
                        "fips_x":"origin",
                        "fips_y":"destination",
                        "LATITUDE_x":"origin_lat",
                        "LONGITUDE_x":"origin_long",
                        "LATITUDE_y":"dest_lat",
                        "LONGITUDE_y":"dest_long"})
                    )[["origin", "destination", "origin_lat", "origin_long", "dest_lat", "dest_long"]]

county_distances['distance'] = haversine_distance(county_distances['origin_lat'],
                                           county_distances['origin_long'],
                                           county_distances['dest_lat'],
                                           county_distances['dest_long']
                                           )

# remove self loops
county_distances = county_distances.loc[county_distances.origin != county_distances.destination,:]

In [34]:
# big dataset... write to parquet
county_dist_formatted = county_distances.loc[:,["origin","destination","distance"]]
county_dist_formatted.to_parquet('../data/county_populationcenters_distance.parquet', index=False)

### Process (county) patchflow datasets
Reformat unnormalized flow datasets for total (bidirectional) flow between states and organize into 3 column format needed for STEpPE to make spatial embeddings

In [39]:
# process data using the get_total_flow function used for the state level data
rad_constants = [0.01, 0.02, 0.05, 0.1, 0.2, 0.5]
for r in rad_constants:
    src = "../data/patchflow_raw/USA_admin2_radiation_constant_{:.02f}.csv".format(r)
    patch_df = pd.read_csv(src).drop("time", axis=1)
    flow_data = get_total_flow(patch_df, county_conversion_dict, clean_codes=False)
    # filter to only origins/destinations included in the gadm to fips converter
    flow_data = flow_data[flow_data['origin'].isin(gadm2fips["fips"]) & flow_data['destination'].isin(gadm2fips["fips"])]
    path = "../data/flows_processed/USA_admin2_radiation_constant_{:.02f}_processed.parquet".format(r)
    # big datasets... save to parquet
    flow_data.to_parquet(path, index=False)
